# Slotfill: end-to-end on one Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aous1998/slotfill/blob/main/notebooks/slotfill_colab.ipynb)

**Runtime → Change runtime type → T4 GPU** (free tier works), then run top to bottom.

1. Setup, tests and label audit
2. Train the main LoRA adapter (r=16) and the ablation adapter (100 training docs)
3. **Restart the session**, serve the base model + both adapters with vLLM
4. Run the prompted Claude baseline (needs an `ANTHROPIC_API_KEY` Colab secret)
5. Score everything on the same 200 held-out receipts

Training and serving run in separate sessions because Unsloth and vLLM pin different library versions.
Files on `/content` survive a session restart.

## 1. Setup

In [ ]:
!git clone -q https://github.com/aous1998/slotfill.git /content/slotfill
%cd /content/slotfill
!pip install -q -r requirements.txt
!python -m pytest -q

In [ ]:
# The split is committed and frozen (its sha256 is checked by the tests).
# Re-run the audit to see how much of each gold field is recoverable from the OCR text.
!python scripts/audit_labels.py

## 2. Train

~10-15 min per adapter on a T4. Logged per run in `adapters/<name>/train_log.json`:
rank, alpha, learning rate, epochs, data counts, GPU, peak memory, loss history.

In [ ]:
!pip install -q unsloth

In [ ]:
!python scripts/train.py --rank 16 --alpha 32 --lr 2e-4 --epochs 3 --out adapters/slotfill-r16

In [ ]:
# Ablation: same settings, only 100 training receipts. Answers "how much labelling do we need?"
!python scripts/train.py --rank 16 --alpha 32 --lr 2e-4 --epochs 3 --n-train 100 --out adapters/slotfill-n100

In [ ]:
import json, glob
for path in sorted(glob.glob("adapters/*/train_log.json")):
    log = json.load(open(path))
    evals = [h["eval_loss"] for h in log["log_history"] if "eval_loss" in h]
    c = log["config"]
    print(f"{path.split('/')[1]:16s} r={c['rank']} alpha={c['alpha']} lr={c['lr']} docs={log['train_docs']} "
          f"time={log['train_seconds']/60:.1f}min dev_loss/epoch={[round(e, 3) for e in evals]}")

## 3. Serve with vLLM

**Runtime → Restart session now**, then continue from here (no need to re-run the cells above).

In [ ]:
%cd /content/slotfill
!pip install -q vllm -r requirements.txt

In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(["bash", "scripts/serve_vllm.sh"], stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        print(urllib.request.urlopen("http://localhost:8000/v1/models").read().decode()[:300])
        break
    except Exception:
        time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

In [ ]:
# What the GPU costs per hour if you rented it (a T4 is roughly $0.35-0.45/h on RunPod/Vast; an L4 about $0.70).
GPU_HOURLY = 0.40
BASE = "Qwen/Qwen2.5-3B-Instruct"
!python scripts/predict.py --provider vllm --model {BASE}           --gpu-hourly {GPU_HOURLY} --out results/preds_base.jsonl
!python scripts/predict.py --provider vllm --model slotfill-r16     --gpu-hourly {GPU_HOURLY} --out results/preds_lora_r16.jsonl
!python scripts/predict.py --provider vllm --model slotfill-n100    --gpu-hourly {GPU_HOURLY} --out results/preds_lora_n100.jsonl

## 4. Prompted frontier baseline

Add `ANTHROPIC_API_KEY` in Colab's 🔑 Secrets panel. 200 receipts on Claude Opus 5.5 cost roughly $1-4 per run
(exact cost is computed from the token counts the API returns).

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# Same prompt as the small models, free-form output:
!python scripts/predict.py --provider anthropic --model claude-opus-5-5 --out results/preds_claude.jsonl
# Same prompt, decoding constrained to the JSON schema (the strongest version of "just use the prompt"):
!python scripts/predict.py --provider anthropic --model claude-opus-5-5 --structured --out results/preds_claude_structured.jsonl

## 5. Results

In [ ]:
!python scripts/evaluate.py \
  --run "Claude Opus 5.5, prompted=results/preds_claude.jsonl" \
  --run "Claude Opus 5.5, structured output=results/preds_claude_structured.jsonl" \
  --run "Qwen2.5-3B base, prompted=results/preds_base.jsonl" \
  --run "Qwen2.5-3B + LoRA r16 (424 docs)=results/preds_lora_r16.jsonl" \
  --run "Qwen2.5-3B + LoRA r16 (100 docs)=results/preds_lora_n100.jsonl"

In [ ]:
# Look at what each system gets wrong before writing the recommendation.
import json
for line in list(open("results/errors_qwen2_5_3b_lora_r16_424_docs.jsonl"))[:5]:
    e = json.loads(line)
    print(e["id"], e["error"], e["wrong"], "\n  gold:", e["gold"], "\n  got: ", e["raw_output"], "\n")

In [ ]:
# Keep the results: download them and commit results/ to your repo.
!zip -qr slotfill_results.zip results adapters/*/train_log.json adapters/*/adapter_config.json
from google.colab import files
files.download("slotfill_results.zip")